## `traceroute` — Hop-by-Hop Path Diagnosis
[Video lesson](https://www.youtube.com/watch?v=up3bcBLZS74&t=60s)

**Windows**: `tracert example.com`

**Linux**: `traceroute example.com`

`ping` tells you *if* a host is reachable in general. `traceroute` pings **each router** on its way to the destination. `traceroute` tells you *where it breaks* along the way. When a ticket says "I can reach X but not Y" or "connection is slow to this server," `traceroute` shows every router hop between you and the destination, so you can pinpoint whether the problem is your network, an ISP, a specific hop, or the destination itself.

**Developer bridge:** think of it like a stack trace for a network path. Each "frame" is a router (hop) instead of a function call. `traceroute` works by sending packets with increasing **TTL** (Time To Live) values — TTL=1 dies at the first router, which sends back an ICMP "time exceeded" message revealing itself. TTL=2 dies at the second router, and so on, building the path hop by hop.

**COMMAND:** `traceroute`

**PURPOSE (support):** `ping` splits a ticket into "network problem" vs "app problem." `traceroute` goes one level deeper — *where in the network* the problem sits. Essential for escalation: knowing whether to hand a ticket to your own team, the ISP, or the destination's network admins.

**SYNTAX:** `traceroute [options] host [packetsize]`

TCP/IP command. Trace route taken by packets to reach network host. `traceroute` attempts tracing by launching UDP probe packets with a small TTL (time-to-live), then listening for an ICMP “time exceeded” reply from a gateway.
- `host` is the destination hostname or the IP number of the host to reach.
- `packetsize` is the packet size in bytes of the probe datagram. Default is 40 bytes.

**FLAGS:**

|Flag|Meaning|
|---|---|
|`-n`|don't resolve hostnames (numeric IPs only) "don't do a reverse-DNS lookup on each hop's IP to print its hostname — just show me the raw number." — much faster, good first pass|
|`-m <hops>`|max hops to try (default 30)|
|`-w <sec>`|wait time per probe before timeout|
|`-I`|use ICMP echo instead of UDP (like `ping`) — some firewalls only allow this|
|`-T`|use TCP SYN packets — useful when UDP/ICMP is firewalled but a TCP port (e.g. 443) is open|

Compare without `-n` (default behavior does try to resolve each hop):
```shell
$ traceroute google.com
 1  _gateway (172.30.240.1)  1.120 ms
 5  some-isp-router.example.net (88.214.102.228)  136 ms
```
vs with `-n`:
```shell
$ traceroute -n google.com
 1  172.30.240.1  1.120 ms
 5  88.214.102.228  136 ms
```
Same trace, same destination — the only difference is column 2 showing a hostname-plus-IP vs just the IP.

💡 Why: reverse DNS lookups on every single hop add real delay (each hop = one extra DNS round-trip before that line prints), and half the time intermediate ISP routers don't even have a clean PTR record, so you get slow, ugly, sometimes-broken output. `-n` skips all of that.

🔁 Support rule: **`-n` = faster, cleaner troubleshooting output. It never restricts your input — you can always type a hostname as the target, with or without `-n`.**

**EXAMPLE:**

Send 3 data packets to each router on its way to the destination and each time the data packate reaches a router on its path, the router will send back the three data packets to our computer and tell us information about that router, such as the router's IP address and it'ss also tell us the round trip time, measured in `msc` that the three data packets took. And now the **trace is complete**.

In [ ]:
$ traceroute -n google.com
traceroute to google.com (142.250.premium.ip), 30 hops max, 60 byte packets
 1  172.20.0.1  0.451 ms  0.398 ms  0.372 ms
 2  10.0.0.1  1.203 ms  1.150 ms  1.102 ms
 3  * * *
 4  203.0.113.9  8.442 ms  8.401 ms  8.355 ms
 5  142.250.183.14  9.100 ms  9.050 ms  9.011 ms

**READ THE OUTPUT:**

- Each numbered line = one hop (step) that the route took to the destination, with 3 round-trip times (3 probes sent per hop).
- The first hop was fast since it was **within LAN**. The next hops reached the internet the round trip times significantly increase. And the further that data packets have to travel to each router, then **naturally** the round trip times will gradually increase, with the final destination being the **longest trip time**. Like `ping` displays only the time of the final destination.
- The last column tell us IP addresses of **each router** and the final destination, and also tell us the **domain name** if it's available.
- `* * *` = no response from that hop — could mean the router drops ICMP (common, often harmless) or a real dead end. Only worry if it's the *last* line and never resolves.
- **One of the main thing - Consistent Round-Trip Times**: If **latency jumps massively between two consecutive hops** (e.g. 8ms → 200ms) and stays high for every hop after, that's usually **where the slowness is introduced** — not necessarily *that* hop being broken, but that's where you'd **start escalating** (e.g., to your ISP or the destination's network team). `ping` can tell that the round-trip takes linger time, but `traceroute` can pinpoint **where the problem IS**.
- If the trace stops completely partway and never reaches the destination, that's a routing/firewall block — the **last hop you *did* reach** is your escalation point.

> The `traceroute` is a great tool to **pinpoint bottlenecks** and interraptions on a network.

**TICKET SCENARIO:** "I can ping the server but the app times out" or "Site X is slow only from our office."
> First step: `ping` to rule out total outage, then `traceroute` to see if there's a slow/dead hop between you and the destination.

**DIAGNOSIS PATH:**

1. `ping` first — confirm basic reachability and get a baseline latency.
2. `traceroute -n <host>` — see the path, look for **where latency spikes** or **hops go dark**.
3. If it dies at your own first few hops → local network issue.
4. If it dies mid-path at an ISP-looking IP → escalate to ISP/network team, not the app team.
5. If it reaches destination fine but the *app* still times out → not a network path issue, likely app/firewall port block (pivot to `nc -zv` or `curl`).

**GOTCHA (WSL-specific):** WSL2 `traceroute` often shows `* * *` for the first couple of hops because the Hyper-V virtual switch and your Windows host firewall can silently drop ICMP time-exceeded replies. Don't panic if hop 1-2 are blank in WSL but hops beyond that resolve fine — that's a WSL/Windows firewall quirk, not a real network problem. Also: `traceroute` may not be installed by default — if you get `command not found`, run `sudo apt install traceroute`.

> One more gotcha: many public sites block ICMP/UDP traceroute probes entirely past their edge, showing `* * *` all the way to the end even though the site works fine in a browser — that's normal, not a fault.

**TRY NOW:**

Run these in your WSL terminal and paste the real output into the code cells below.

```
traceroute -n 8.8.8.8
traceroute -n google.com
```

Notice hop 1 in each — that's your WSL virtual router/gateway, not your real home router.

In [ ]:
dennie@X390:~/linux-tut$ traceroute -n 8.8.8.8
traceroute to 8.8.8.8 (8.8.8.8), 30 hops max, 60 byte packets
 1  172.30.240.1  1.120 ms  0.794 ms  0.652 ms
 2  * 192.168.47.184  31.991 ms  35.554 ms
 3  * * *
 4  * * *
 5  88.214.102.228  136.249 ms  141.431 ms  139.284 ms
 6  172.17.111.1  141.703 ms  141.310 ms  141.241 ms
 7  * * *
 8  88.214.120.2  113.547 ms  78.754 ms  93.533 ms
 9  192.178.248.235  90.947 ms 192.178.108.169  91.322 ms 192.178.248.235  98.758 ms
10  74.125.245.86  108.462 ms * *
11  * * *
12  * 192.178.99.101  93.872 ms *
13  209.85.253.225  181.076 ms 108.170.234.247  183.870 ms 142.250.239.81  180.819 ms
14  216.239.40.213  182.190 ms 172.253.68.31  184.026 ms 8.8.8.8  182.534 ms

Read both. Solid results — let me walk through what they show.

**Trace 1 (`8.8.8.8`):** Reaches the destination cleanly at hop 14 (`182ms`), and hop 1 is `172.30.240.1` — exactly the WSL virtual gateway called out in the lesson. Hop 2 (`192.168.47.184`) is your real home router.

Between hop 2 (~5ms) and hop 5 (`88.214.102.228`, ~136ms) there's the **big latency jump** the lesson tells you to watch for — that's where your ISP's backbone picks up the traffic and adds real distance. So, the problem is **OUT ON THE INTERNET** on the fifth hop starting with this router and affecting the remaining path to the server. So we **PINPOINTED** that the slow connection or lag in **NOT** within our LAN or with the server. The problem is with a router or routers **ON THE INTERNET**.

Sometimes you might see **high round-trip** compared to the other two **on the same hop**. This is not a big deal, it just means that something strange has happened to that data packet. Because if ALL THREE were high than that indicates an issue.

Hops 3, 4, 7, 11 **going dark** (`* * *`) mid-path but the trace still completing is the classic "routers along the way drop ICMP but still forward traffic fine" case — not a fault. It means, the **router is working fine**, but it **wasn't configured** to return traceroute replies, but the router still **passes on the data packets** to the next router.

Sometimes, high round-trip times while doing a traceroute doesn't always indicate a problem. If you happen to see a big round-trip time in between hops it doesn't necessarily mean that the problem is with the router. It could just mean that the **distance between** certain routers is thousands of kilometers apart, for example, when a data has to jump to another continent.

**TTL (time to live)**: it's a given value to the packets on how long they can live before they are discarded. `30 hops max` means TTL is set to `30` which is the default value. So, if the data packets don't reach their destination after 30 hops the data packets are dropped. Below we set TTL to 4 hops:

In [ ]:
dennie@X390:~/linux-tut$ traceroute -m 4 google.com
traceroute to google.com (142.251.98.138), 4 hops max, 60 byte packets
 1  X390.mshome.net (172.30.240.1)  1.198 ms  0.941 ms  0.689 ms
 2  192.168.47.184 (192.168.47.184)  3.476 ms  3.440 ms  3.710 ms
 3  * * *
 4  * * *

This means, the data reaches the 4th hop, the data packet is dropped and will not continue on.

> **Purpose of TTL**: TTL prevents data packets from traveling **endlessly** around the internet thying to find its destination. This can happen if certain routers on the internet were misconfigured. The data could be caught in an endless loop between **misconfigured routers** and would just keep passing these data packets back to each other forever. This is why the time limit was placed to keep this from happening.

In [ ]:
dennie@X390:~/linux-tut$ traceroute -n google.com
traceroute to google.com (142.250.109.138), 30 hops max, 60 byte packets
 1  172.30.240.1  1.767 ms  1.590 ms  1.303 ms
 2  192.168.47.184  4.438 ms  5.470 ms  4.338 ms
 3  * * *
 4  * * *
 5  88.214.102.204  96.759 ms  96.726 ms  105.155 ms
 6  * * *
 7  * * *
 8  209.85.168.34  100.089 ms  79.466 ms  87.609 ms
 9  192.178.108.169  95.946 ms 192.178.248.235  98.405 ms 192.178.108.169  80.966 ms
10  74.125.245.86  90.231 ms 74.125.245.84  94.085 ms 74.125.245.62  124.300 ms
11  72.14.239.110  130.239 ms 72.14.239.111  113.108 ms 142.251.242.41  121.984 ms
12  216.239.41.1  115.063 ms 192.178.122.208  100.304 ms 192.178.99.169  100.506 ms
13  * 192.178.247.52  117.833 ms 192.178.99.177  104.751 ms
14  192.178.46.144  104.419 ms * *
15  * * *
16  * * *
17  * * *
18  * * *
19  * * *
20  * * *
21  * * 142.250.109.138  197.508 ms

**Trace 2 (`google.com` → `142.250.109.138`):** Same hop 1/2 as expected. Then hops 15 through 20 go **completely dark** — six blank hops in a row — before hop 21 suddenly shows a real reply _from the actual destination IP_. That's a textbook example of the gotcha in the lesson: Google's edge network is silently dropping/not responding to intermediate TTL-expired probes, but the destination itself still answers the final one. If this were a real ticket, seeing 6 stars in a row would look alarming, but the correct read is: **the trace still resolved at the end, so the path works — that block is deliberate destination-side filtering, not a break.**

One thing to flag: notice `google.com` took 21 hops vs `8.8.8.8`'s 14 — different Google Cloud infrastructure/edge PoP being used, not a problem, just a reminder that hostname vs raw IP can genuinely take different real-world paths.

Verdict: both traces are healthy, and you've now seen live examples of both gotchas from the lesson (mid-path silence + edge filtering) in the same session. Nice pair of results for pattern-matching later.

For deeper theory on this, check the freeCodeCamp course's "Circuit vs. Packet Switching" (11:36:16) — helps explain _why_ each hop can take a different path per probe (you can see this in trace 1, hop 9, where three different IPs answered the three probes).

---

It's interesting, since I use my smartphone with the internet connection to connect my laptop where I ran these commands. How to interpret the 'your real home router'? Is it my smartphone IP instead of router? It acts like a router? Plus to the context, I'm driving in the car, so the 'going dark' means sometimes along the way the connection drops and this reflects in '*', right?

Good instinct to question that — let's separate the two questions.

**Is that your phone?**

Yes. `192.168.47.184` is a private IP, and since your laptop's internet path is phone → WSL, that address belongs to **your phone's mobile hotspot interface**. Functionally, your phone _is_ acting as a router at that point:

- It does **NAT** (translates your laptop's private IP to the phone's cellular IP going out to the carrier).
- It hands out a DHCP lease (that's how your laptop got an address on the `192.168.47.x` network in the first place).
- It forwards packets between your laptop and the cellular radio.

That's literally what a router does — hardware router vs phone hotspot is just where those functions live, not a difference in function. **Developer bridge:** think of your phone here like a reverse proxy doing NAT — same job as your home router, different box.

So your actual chain per trace is: WSL virtual gateway (hop 1) → phone hotspot NAT (hop 2) → cell tower → carrier backbone (hop 5 onward, that's the `88.214.x.x` jump).

**Is "going dark" the connection dropping while driving?**

Not necessarily — and this is an important distinction to keep straight for tickets. A `*` means **that one probe got no reply within the timeout window (default ~5s)**, for one of two very different reasons:

1. **Deliberate filtering** — the router at that hop is configured to not respond to TTL-expired ICMP, but it forwards traffic through it just fine. This is what the lesson's gotcha covers, and it's the _most common_ reason for stars on a stable wired connection.
2. **Real transient loss** — a packet actually got dropped or delayed past the timeout. On a moving cellular connection this is completely plausible: tower handoffs, signal fade, congestion — all real events, not filtering.

Since you were driving, both are in play simultaneously, and a single trace can't tell you which caused which star. **The way to tell them apart:** run the same `traceroute -n 8.8.8.8` again from a stationary position (parked, same phone/hotspot). If the _same_ hops go dark every time → that's filtering (structural, hop-specific). If _different_ hops go dark on each run, or it clears up entirely once stationary → that was real motion-induced loss, not the router's fault.

**TICKET SCENARIO for this exact situation:** "User on a cellular hotspot reports intermittent connectivity" — this is why you'd never diagnose a mobile/cellular link off one traceroute. You'd want 3-4 runs minimum, ideally comparing moving vs stationary, before concluding whether it's the network path or the radio link itself.

**One more thing worth adding — a third cause of `*`, and the tool built for exactly your scenario.**

There's actually a **third** reason for stars, distinct from the two already covered:

3. **ICMP rate-limiting** — most routers deliberately cap how many "TTL exceeded" replies they'll *generate* per second (a security/performance measure, RFC 4443), completely independent of motion or load. This means a hop can answer probe 1, stay silent on probe 2, then answer probe 3 — on a perfectly stable wired connection, no driving involved. It's different from outright filtering (that hop *never* replies, ever) and different from real loss (packet actually dropped in transit). Rate-limiting is actually the most common reason you'll see isolated single stars mixed in with real replies on ordinary networks.

So the full list is now: **filtering** (never responds, by policy), **rate-limiting** (responds sometimes, by policy), **real loss** (responds sometimes, by accident — motion, congestion, signal fade).

**The tool for telling them apart without manual re-runs: `mtr`.** It's `ping` and `traceroute` combined into one continuously-updating view — instead of one static snapshot, it keeps looping the trace and shows a running loss % *per hop* over time. This is the direct answer to "run it a few times, stationary vs moving" from a minute ago — `mtr` does that repetition for you automatically.

```
sudo apt install mtr   # not installed by default
mtr -rw -c 50 8.8.8.8
```

`-r` = report mode (prints a summary instead of a live TUI — better for pasting into a ticket or a notebook like this one), `-w` = wide output (full hostnames/IPs, no truncation), `-c 50` = send 50 cycles before reporting. Needs raw ICMP, so if you see a permissions complaint, run it as `sudo mtr -rw -c 50 8.8.8.8` instead.

**READ THE OUTPUT:** each hop gets its own `Loss%` column accumulated across all 50 cycles. A hop sitting at a *consistent* non-zero loss% (say 15% every run) points to real, structural loss at that hop — worth escalating. A hop that's 0% most runs but spikes only while you're actually moving points to the cellular link itself, not the network path.

**TICKET SCENARIO:** "Connection is dropping on mobile/hotspot" — before concluding "the network is flaky," reach for `mtr -rw -c 50 <target>` first. One command, run once stationary and once moving, gives you the per-hop loss data to say *where* the flakiness lives instead of guessing from a single traceroute snapshot.